In [ ]:
import json

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validar_test_individual, guardar_i_netejar_tests

In [2]:
g = build_graph_from_directory(code_path="../src", save_graph=False)

In [ ]:
# TODO: filtrar per tipus de node: function or method
# Per a cada node del graf, obtenim el seu context i creem el test
node_id = "utgen/raggraph/utils.py::function::normalize_signature"

# TODO: eliminar nested_functions del context
context = get_node_context(g=g, node_id=node_id)

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Extract metadata
inputs = {"graph_context": context}

response = test_generator.crew().kickoff(inputs=inputs)

# Convert string to dictionary
data = json.loads(response.raw)

In [ ]:
# Validem el test generat
node_id = "utgen/raggraph/utils.py::function::normalize_signature"

split_1 = node_id.split('/')
split_2 = split_1[-1].split('::')
split_3 = split_2[0].split('.')

test_func_import = f"\nfrom {split_1[0]}.{split_1[1]}.{split_3[0]} import {split_2[-1]}"
filename = f"test_{split_2[-1]}.py"

tests_que_han_passat = []

for k, v in data['tests'].items():
    name, imp, code = k, v['imports'], v['code']
    # imp = imp + test_func_import
    if validar_test_individual(imp, code):
        print(f"✅ Test `{name}` acceptat")
        tests_que_han_passat.append((imp, code))
    else:
        print(f"❌ Test `{name}` rebutjat")

# Guardem i deixem el fitxer perfecte
guardar_i_netejar_tests(tests_que_han_passat, desti=f"../tests/{filename}")

# TODO: script final que reordene la carpeta tests: un test.py per cada .py del src

✅ Test `test_normalize_signature_basic_function` acceptat
✅ Test `test_normalize_signature_with_annotations` acceptat
❌ Test `test_normalize_signature_with_vararg_and_kwargs` rebutjat
✅ Test `test_normalize_signature_with_posonly_args` acceptat
✅ Test `test_normalize_signature_with_kwonly_args` acceptat
❌ Test `test_normalize_signature_with_all_arg_types` rebutjat
✅ Test `test_normalize_signature_invalid_input` acceptat
Netejant ../tests/test_normalize_signature.py amb Ruff...
✅ Procés finalitzat. Fitxer guardat a: ../tests/test_normalize_signature.py
